In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
notebook_name = "03_silver_traffic_batch"

In [0]:
def log_pipeline_error(step_name, error):
    spark.createDataFrame(
        [(notebook_name, step_name, str(error), datetime.now())],
        ["notebook", "step", "error_message", "error_time"]
    ).write.mode("append").saveAsTable("dev_catalog.default.pipeline_errors")

In [0]:
df_raw_traffic = spark.table("dev_catalog.bronze.raw_traffic")
df_raw_roads = spark.table("dev_catalog.silver.roads")

In [0]:

df_traffic = (
    df_raw_traffic
    .drop("link_length_km")
    .withColumn("count_point_id", col("count_point_id").try_cast("int"))
    .withColumn("year", col("year").try_cast("int"))
    .withColumn("pedal_cycles", col("pedal_cycles").try_cast("int"))
    .withColumn("two_wheeled_motor_vehicles", col("two_wheeled_motor_vehicles").try_cast("int"))
    .withColumn("cars_and_taxis", col("cars_and_taxis").try_cast("int"))
    .withColumn("buses_and_coaches", col("buses_and_coaches").try_cast("int"))
    .withColumn("LGVs", col("LGVs").try_cast("int"))
    .withColumn("HGVs_2_rigid_axle", col("HGVs_2_rigid_axle").try_cast("int"))
    .withColumn("HGVs_3_rigid_axle", col("HGVs_3_rigid_axle").try_cast("int"))
    .withColumn("HGVs_4_or_more_rigid_axle", col("HGVs_4_or_more_rigid_axle").try_cast("int"))
    .withColumn("HGVs_3_or_4_articulated_axle", col("HGVs_3_or_4_articulated_axle").try_cast("int"))
    .withColumn("HGVs_5_articulated_axle", col("HGVs_5_articulated_axle").try_cast("int"))
    .withColumn("HGVs_6_articulated_axle", col("HGVs_6_articulated_axle").try_cast("int"))
    .withColumn("all_HGVs", col("all_HGVs").try_cast("int"))
    .withColumn("link_length_miles", col("link_length_miles").try_cast("double"))
    .withColumn("all_motor_vehicles", col("all_motor_vehicles").try_cast("int"))
    .withColumn("transformed_time", current_timestamp())
    .withColumn("source" , ifnull(col("source"),lit("A")))
)


In [0]:
from pyspark.sql.functions import broadcast
df_final = (
   
    df_traffic.join(df_raw_roads.select("count_point_id", "link_length_km" ), on="count_point_id", how="left")
    .dropDuplicates(['count_point_id' , "direction_of_travel"])
    .withColumn("recomputed_motor_vehicles" , col('buses_and_coaches') + col('cars_and_taxis') + col('two_wheeled_motor_vehicles') + col('LGVs') + col('HGVs_2_rigid_axle') + col('HGVs_3_rigid_axle') + col('HGVs_4_or_more_rigid_axle') + col('HGVs_3_or_4_articulated_axle') + col('HGVs_5_articulated_axle') + col('HGVs_6_articulated_axle'))
    .withColumn("vehicle_count_consistent" , when(col("all_motor_vehicles") == col("recomputed_motor_vehicles"), True).otherwise(False))
    .withColumn("total_traffic_volume" , col('all_motor_vehicles') + col('pedal_cycles'))
    .withColumn("vehicle_intensity" , when((col('link_length_km').isNull()) | (col('link_length_km') == 0),lit(None))
                .otherwise(col("all_motor_vehicles") / col('link_length_km')))
    .withColumn("is_estimated" , when(col("estimation_method") == "Estimated", True).otherwise(False))
    .withColumn("transformed_time" , current_timestamp())
)
    

In [0]:
df_final.printSchema

In [0]:
df_final.createOrReplaceTempView("traffic_updates")

In [0]:
%sql
MERGE INTO dev_catalog.silver.traffic AS target
USING traffic_updates AS src
ON target.count_point_id = src.count_point_id 
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
